# PulseMix Recommendation Notebook

This notebook documents the end-to-end recommendation workflow:

1. Load and inspect the raw music feature dataset
2. Perform compact EDA
3. Build the content-based recommendation pipeline
4. Evaluate the ML and deep-learning layers
5. Show where collaborative and hybrid models fit once interaction data is available


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from music_recommendation.config import get_config
from music_recommendation.data.loaders import load_content_data, load_interaction_data
from music_recommendation.features.preprocessing import build_content_features
from music_recommendation.models.content_based import ContentBasedRecommender
from music_recommendation.models.deep_models import DeepContentTrainer
from music_recommendation.models.ml_models import YearRegressionBaseline
from music_recommendation.pipelines.training import run_training_pipeline

sns.set_theme(style="whitegrid")
config = get_config()


## Load Data

In [ ]:
content_dataset = load_content_data(config)
interaction_dataset = load_interaction_data(config)

content_dataset.frame.head()

In [ ]:
content_dataset.frame.shape, len(content_dataset.feature_columns), interaction_dataset is not None

## EDA

The available dataset is item-level, not user-level. That makes it ideal for feature-driven recommendation and representation learning.

In [ ]:
eda_frame = content_dataset.frame[["year"] + content_dataset.feature_columns[:8]].copy()
eda_frame.describe().T.head(10)

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(content_dataset.frame["year"], bins=30, color="#ef8354")
plt.title("Distribution of Song Years")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(eda_frame.corr(numeric_only=True), cmap="mako")
plt.title("Correlation Snapshot")
plt.show()

## Content-Based Recommendations

In [ ]:
bundle = build_content_features(content_dataset, latent_dims=config.train.latent_factors)
content_model = ContentBasedRecommender(top_k=config.dataset.top_k).fit(content_dataset, bundle)
seed_item = content_dataset.frame.iloc[0]["track_id"]
content_model.recommend(seed_item).head(10)

## Classical ML Layer

In [ ]:
ml_result = YearRegressionBaseline().fit_and_evaluate(content_dataset)
ml_result

## Deep Learning Layer

In [ ]:
dl_result = DeepContentTrainer(
    epochs=config.train.epochs,
    batch_size=config.train.batch_size,
    learning_rate=config.train.learning_rate,
).fit(content_dataset)
dl_result

## Full Pipeline Run

In [ ]:
results = run_training_pipeline(config)
results

## Note on Collaborative Filtering and Hybrid Ranking

These modules are already scaffolded in the codebase, but they require a second dataset containing `user_id`, `track_id`, and `rating` (or implicit feedback) to train and evaluate properly.